# Experimentos CUDA en Google Colab

**Proyecto:** Sistema de Predicción de Demanda para el Comercio Minorista Fronterizo (ADR-005).

El equipo de desarrollo no tiene GPU NVIDIA, así que el entrenamiento con CUDA se ejecuta aquí.
Colab **no reemplaza a CUDA**: PyTorch usa CUDA sobre la GPU de Google.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

**Regla de medición:** la aceleración se calcula comparando CPU y GPU **en esta misma sesión**
(nunca la CPU del equipo local contra la GPU de Colab). Los datos son **sintéticos**: sirven para
medir rendimiento, no para concluir nada sobre la demanda real.

## 1. Verificar la GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repositorio e instalar el núcleo analítico

In [ ]:
REPO_URL = "https://github.com/owen854/Sistema-Prediccion-Comercio.git"
RAMA = "main"  # cambie a la rama de trabajo si es necesario

!rm -rf /content/repo
!git clone --depth 1 --branch {RAMA} {REPO_URL} /content/repo
# Colab ya trae PyTorch con CUDA: no se instala el extra "dl".
!pip install -q -e "/content/repo/Codigo Fuente/backend"

In [ ]:
from forecast_core.hpc.cuda import cuda_info
from forecast_core.hpc.benchmark import hardware_info

info = hardware_info()
for k, v in info.items():
    print(f"{k:>22}: {v}")
assert info["gpu_cuda_available"], "No hay GPU: cambie el tipo de entorno a GPU y vuelva a ejecutar."

## 3. *Spike*: entrenamiento LSTM multiserie, CPU frente a GPU

Se entrena una LSTM pequeña sobre ventanas de 28 días de muchas series sintéticas a la vez
(el escenario donde la GPU aporta, según la revisión sistemática). La LSTM definitiva del
sistema se implementa en `forecast_core.models.lstm` (Fases 2 y 3).

In [ ]:
import json, time
import numpy as np
import torch
from forecast_core.ingestion import synthetic
from forecast_core.preprocessing.panel import to_panel
from forecast_core.tracking.seeds import set_global_seed

N_SERIES, DIAS, VENTANA = 200, 730, 28
OCULTAS, EPOCAS, LOTE = 32, 5, 4096

ventas, _ = synthetic.generate(n_series=N_SERIES, start="2023-01-01", end="2024-12-30", seed=42)
panel = to_panel(ventas)
Y = np.log1p(panel.values)  # (T, N)

def ventanas(Y, W):
    T, N = Y.shape
    idx = np.arange(W)[None, :] + np.arange(T - W)[:, None]  # (T-W, W)
    X = Y[idx].transpose(2, 0, 1).reshape(-1, W, 1)          # (N*(T-W), W, 1)
    y = Y[W:].T.reshape(-1, 1)
    return X.astype(np.float32), y.astype(np.float32)

X, y = ventanas(Y, VENTANA)
print("Origen de datos:", panel.origin.value, "| ventanas:", X.shape)

class RedLSTM(torch.nn.Module):
    def __init__(self, ocultas):
        super().__init__()
        self.lstm = torch.nn.LSTM(1, ocultas, batch_first=True)
        self.salida = torch.nn.Linear(ocultas, 1)
    def forward(self, x):
        h, _ = self.lstm(x)
        return self.salida(h[:, -1])

def entrenar(dispositivo):
    set_global_seed(0)
    modelo = RedLSTM(OCULTAS).to(dispositivo)
    opt = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    Xt, yt = torch.from_numpy(X).to(dispositivo), torch.from_numpy(y).to(dispositivo)
    n = Xt.shape[0]
    if dispositivo == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(EPOCAS):
        orden = torch.randperm(n, device=dispositivo)
        for i in range(0, n, LOTE):
            j = orden[i:i + LOTE]
            opt.zero_grad()
            perdida = torch.nn.functional.mse_loss(modelo(Xt[j]), yt[j])
            perdida.backward()
            opt.step()
    if dispositivo == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t0, float(perdida)

t_gpu, l_gpu = entrenar("cuda")
t_cpu, l_cpu = entrenar("cpu")
resultado = {
    "experimento": "spike_cuda_lstm_multiserie",
    "origen_datos": panel.origin.value,
    "series": N_SERIES, "ventanas": int(X.shape[0]), "epocas": EPOCAS, "lote": LOTE,
    "tiempo_cpu_s": t_cpu, "tiempo_gpu_s": t_gpu, "aceleracion_gpu": t_cpu / t_gpu,
    "perdida_final_cpu": l_cpu, "perdida_final_gpu": l_gpu,
    "hardware": info,
}
print(json.dumps({k: v for k, v in resultado.items() if k != "hardware"}, indent=2))

## 4. Guardar el resultado

In [ ]:
import os
from datetime import datetime, timezone

os.makedirs("/content/resultados", exist_ok=True)
nombre = f"spike_cuda_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.json"
ruta = f"/content/resultados/{nombre}"
with open(ruta, "w", encoding="utf-8") as fh:
    json.dump(resultado, fh, indent=2, ensure_ascii=False)
print("Guardado en", ruta)

# Opcional: copiar a Google Drive para no perderlo al cerrar la sesión.
# from google.colab import drive; drive.mount("/content/drive")
# !cp {ruta} "/content/drive/MyDrive/"

from google.colab import files
files.download(ruta)  # luego se copia a "Codigo Fuente/experiments/results/spike/"